<a href="https://colab.research.google.com/github/ErrPeppe/OhnomiTools/blob/main/notebooks/01_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1 — Acquisizione Dati, Normalizzazione Genomica e Pipeline ETL (`Bronze` $\to$ `Silver`)

## Introduzione Biologica, Fisica e Metodologica

### 1. Contesto Biologico e Fisico: Duplicazioni Genomiche e Bilanciamento del Dosaggio
Alla base della radiazione dei vertebrati (circa 500 milioni di anni fa), il genoma ancestrale ha subito due eventi successivi di duplicazione dell'intero genoma (**1R/2R Whole Genome Duplications**, o **WGD**), ipotizzati originariamente da Susumu Ohno (1970). I geni paraloghi sopravvissuti a questi eventi macro-evolutivi — chiamati **ohnologhi** — mostrano proprietà radicalmente diverse rispetto ai geni originati da duplicazioni locali su piccola scala (**Small Scale Duplications**, o **SSD**).

Dal punto di vista della fisica statistica dei sistemi biologici, questa asimmetria è spiegata dall'**ipotesi del bilanciamento del dosaggio (*dosage balance*)**:
* una **SSD** altera i rapporti stechiometrici dei complessi macromolecolari e delle reti regolatorie, risultando spesso svantaggiosa se non compensata da una rapida riduzione di espressione o divergenza funzionale;
* una **WGD**, duplicando simultaneamente l'intero genoma, preserva intatti tutti i rapporti stechiometrici, favorendo la ritenzione selettiva di geni altamente connessi e dosaggio-sensibili (fattori di trascrizione, proteine di segnalazione e RNA regolatori).

I lavori del gruppo di Fisica dei Sistemi Complessi dell'Università di Torino hanno dimostrato l'impronta delle 1R/2R-WGD prima sui geni codificanti e sulle reti trascrizionali (Mottes et al., *PLOS Comput. Biol.* 2021) e successivamente sui microRNA intragenici (Agasso et al., *PLOS Comput. Biol.* 2025). **L'obiettivo di questo lavoro è estendere tale impianto quantitativo ai *long non-coding RNA* (lncRNA) umani**, costruendo al contempo una libreria Python open-source e riproducibile (`OhnomiTools`).

### 2. Selezione delle Sorgenti Dati
Per garantire coerenza degli identificativi (evitando disallineamenti tra release genomiche) e permettere un confronto rigoroso tra origine WGD e controllo SSD, questo notebook integra tre sorgenti primarie su genoma di riferimento **GRCh38**:
1. **LncBook v2.0** (Li et al., *Nucleic Acids Research* 2023): catalogo curato e multi-omico dei lncRNA umani, scelto come asse portante per annotazione genomica (`.gtf.gz` puro), filostratigrafia su 40 vertebrati (`conservation`), rete di interazioni lncRNA–miRNA ad alta stringenza (intersezione dei tre algoritmi termodinamici *miRanda*, *TargetScan* e *RNAhybrid*) ed espressione tissutale (`expression`).
2. **OHNOLOGS (`homo_sapiens`):** dump completo del database di riferimento degli ohnologhi umani derivati dalle 1R/2R-WGD (`ohnologs/homo_sapiens.tsv`, `database/genes.tsv`, `database/gene_ohnology.tsv`).
3. **Ensembl BioMart (Archivi HTTPS ufficiali):** estratto del catalogo dei geni umani `protein_coding` (necessario per ottenere il filamento `strand` `+`/`-` non presente nei file OHNOLOGS) e dell'insieme completo dei paraloghi intra-umani da cui ricavare il gruppo di controllo **SSD**.

### 3. Metodi Computazionali e Data Engineering
* **Architettura Cloud Riproducibile:** codice sorgente versionato su GitHub (`ErrPeppe/OhnomiTools`), calcolo eseguito su Google Colab e persistenza dei dati su Google Drive secondo un'architettura **Medallion** (`bronze/` dati grezzi immutabili $\to$ `silver/` tabelle colonnari normalizzate $\to$ `gold/` tabelle statistiche e figure).
* **Formato Colonnare e Motore Vettorizzato:** tutti i file grezzi (`.gtf.gz`, `.csv.gz`, `.tsv`) vengono convertiti tramite **Polars** in formato **Apache Parquet** compresso con algoritmo **Zstandard (`zstd`)**, utilizzando scritture atomiche tramite cache SSD locale (`/tmp`) per evitare colli di bottiglia sul filesystem FUSE di Drive.
* **Normalizzazione e Costruzione del Controllo SSD Puro:**
  * tutti gli insiemi genomici vengono filtrati sui soli **cromosomi canonici** ($\text{chr1}\text{–}\text{chr22}, \text{chrX}, \text{chrY}$) con prefisso unificato `chr`;
  * i file TSV di OHNOLOGS (privi di intestazione e indicizzati su identificativi proteici `ENSP` multi-specie) vengono filtrati su `homo_sapiens` e tradotti in identificativi genici stabili `ENSG` con schema relazionale esplicito;
  * ogni coppia non orientata $\{g_1, g_2\}$ viene mappata su una chiave canonica ordinata lessicograficamente:
    $$k(g_1, g_2) = \min(g_1, g_2) \mathbin{\Vert} \text{"\_"} \mathbin{\Vert} \max(g_1, g_2)$$
    che elimina duplicati simmetrici $(A,B) \equiv (B,A)$ e permette di definire matematicamente il gruppo di controllo **SSD puro** come differenza insiemistica (*anti-join*):
    $$\mathcal{P}_{\text{SSD}} = \mathcal{P}_{\text{Ensembl, protein\_coding}} \setminus \mathcal{P}_{\text{WGD}}$$

## 1. Inizializzazione dell'Ambiente Cloud (`OhnomiTools` + Google Drive)
Installazione della libreria da GitHub e montaggio automatico della cartella dati su Google Drive tramite `setup_colab()`, con stampa dello stato di presenza e dimensione di tutti i file `bronze` e `silver`.

In [ ]:
!pip install -q --no-cache-dir git+https://github.com/ErrPeppe/OhnomiTools.git
from ohnomitools import setup_colab
PATHS = setup_colab()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
shape: (20, 5)
┌────────┬─────────────────────────────────┬──────────┬─────────┬──────────────────────────────────┐
│ Layer  ┆ Dataset                         ┆ Presente ┆ Size_MB ┆ Path                             │
│ ---    ┆ ---                             ┆ ---      ┆ ---     ┆ ---                              │
│ str    ┆ str                             ┆ str      ┆ f64     ┆ str                              │
╞════════╪═════════════════════════════════╪══════════╪═════════╪══════════════════════════════════╡
│ Bronze ┆ LncBook v2.0 GTF                ┆ SI       ┆ 15.2    ┆ bronze/lncRNA_LncBookv2.0_GRCh38 │
│        ┆                                 ┆          ┆         ┆ .gtf.gz                          │
│ Bronze ┆ LncBook v2.0 Conservation       ┆ SI       ┆ 46.78   ┆ bronze/conservation_LncBook2.0.c │
│        ┆                                

## 2. Esecuzione della Pipeline ETL (`Bronze` $\to$ `Silver`)
Conversione del GTF di LncBook v2.0 (estrazione degli attributi dalla colonna 9), streaming out-of-core dei CSV di conservazione/interazioni/espressione, normalizzazione dei file OHNOLOGS privi di intestazione e download filtrato (`protein_coding`, cromosomi canonici) dei paraloghi umani da Ensembl BioMart.

In [ ]:
from ohnomitools.fetcher import DataIngestionPipeline

pipeline = DataIngestionPipeline(paths=PATHS)
pipeline.fetch_and_build_ensembl_pc_genes(force_rebuild=False)
pipeline.build_ohnologs_wgd_tables(force_rebuild=True)
silver_files = pipeline.run_all(force_rebuild=False)
PATHS.print_status()

shape: (20, 5)
┌────────┬─────────────────────────────────┬──────────┬─────────┬──────────────────────────────────┐
│ Layer  ┆ Dataset                         ┆ Presente ┆ Size_MB ┆ Path                             │
│ ---    ┆ ---                             ┆ ---      ┆ ---     ┆ ---                              │
│ str    ┆ str                             ┆ str      ┆ f64     ┆ str                              │
╞════════╪═════════════════════════════════╪══════════╪═════════╪══════════════════════════════════╡
│ Bronze ┆ LncBook v2.0 GTF                ┆ SI       ┆ 15.2    ┆ bronze/lncRNA_LncBookv2.0_GRCh38 │
│        ┆                                 ┆          ┆         ┆ .gtf.gz                          │
│ Bronze ┆ LncBook v2.0 Conservation       ┆ SI       ┆ 46.78   ┆ bronze/conservation_LncBook2.0.c │
│        ┆                                 ┆          ┆         ┆ sv.gz                            │
│ Bronze ┆ LncBook v2.0 miRNA Interactions ┆ SI       ┆ 1.6     ┆ bronze/lnc

## 3. Controllo Qualità (QC) dei Dataset WGD e SSD
Verifica dei volumi finali estratti in `silver/` e ispezione dello schema delle coppie **WGD** (OHNOLOGS) e **SSD pure** (Ensembl BioMart dopo sottrazione delle WGD), incluse coordinate genomiche, `strand` e clade dell'ultimo antenato comune (LCA).

In [ ]:
import polars as pl

df_wgd_g = pl.read_parquet(PATHS.silver_wgd_genes)
df_wgd_p = pl.read_parquet(PATHS.silver_wgd_pairs)
df_ssd_p = pl.read_parquet(PATHS.silver_ssd_pairs)

print(f"Geni WGD umani (OHNOLOGS)    : {len(df_wgd_g):,}")
print(f"Coppie WGD intra-umane       : {len(df_wgd_p):,}")
print(f"Coppie SSD pure (Ensembl)    : {len(df_ssd_p):,}")
display(df_wgd_p.head(3))
display(df_ssd_p.head(3))

Geni WGD umani (OHNOLOGS)    : 4,602
Coppie WGD intra-umane       : 4,221
Coppie SSD pure (Ensembl)    : 127,578


protein_id_1,protein_id_2,wgd_round,gene_id_1,gene_name_1,chr_1,start_1,end_1,strand_1,family_id_1,retention_category_1,gene_id_2,gene_name_2,chr_2,start_2,end_2,strand_2,family_id_2,retention_category_2,pair_key
str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64,str,str,str,str
"""ENSP00000367830""","""ENSP00000295797""","""unk""","""ENSG00000067606""","""PRKCZ""","""chr1""",2050411,2185395,"""+""","""OG0002838""","""unk""","""ENSG00000163558""","""PRKCI""","""chr3""",170222424,170305977,"""+""","""OG0002838""","""unk""","""ENSG00000067606_ENSG0000016355…"
"""ENSP00000367668""","""ENSP00000478173""","""unk""","""ENSG00000142606""","""MMEL1""","""chr1""",2590639,2633016,"""-""","""OG0000142""",null,"""ENSG00000196549""","""MME""","""chr3""",155024124,155183704,"""+""","""OG0000142""",null,"""ENSG00000142606_ENSG0000019654…"
"""ENSP00000270722""","""ENSP00000498411""","""unk""","""ENSG00000142611""","""PRDM16""","""chr1""",3069168,3438621,"""+""","""OG0002282""",null,"""ENSG00000085276""","""MECOM""","""chr3""",169083499,169663775,"""-""","""OG0002282""",null,"""ENSG00000085276_ENSG0000014261…"


gene_id_1,gene_id_2,Paralogue last common ancestor with Human,homology_type,perc_id_1,perc_id_2,pair_key,gene_name_1,chr_1,start_1,end_1,strand_1,gene_name_2,chr_2,start_2,end_2,strand_2
str,str,str,str,f64,f64,str,str,str,i64,i64,str,str,str,i64,i64,str
"""ENSG00000137177""","""ENSG00000196169""","""Opisthokonta""","""other_paralog""",11.8006,21.3427,"""ENSG00000137177_ENSG0000019616…","""KIF13A""","""chr6""",17759183,17987635,"""-""","""KIF19""","""chr17""",74326188,74355835,"""+"""
"""ENSG00000214940""","""ENSG00000234719""","""Homininae""","""within_species_paralog""",72.3577,67.2544,"""ENSG00000214940_ENSG0000023471…","""NPIPA8""","""chr16""",18317919,18336736,"""-""","""NPIPB2""","""chr16""",11927259,11976643,"""-"""
"""ENSG00000167858""","""ENSG00000092850""","""Bilateria""","""within_species_paralog""",32.5359,31.6279,"""ENSG00000092850_ENSG0000016785…","""TEKT1""","""chr17""",6789133,6831761,"""-""","""TEKT2""","""chr1""",36084094,36088278,"""+"""


## 4. Discussione dei Risultati della Fase 1

L'esecuzione della pipeline ETL (`DataIngestionPipeline`) ha trasformato con successo tutte le sorgenti grezze disponibili nel layer `bronze/` nei corrispondenti dataset colonnari compressi nel layer `silver/` su Google Drive:

* **Compressione e integrità dello storage:** la conversione in Parquet `zstd` ha ridotto drasticamente l'impronta su disco e i tempi di lettura (es. il GTF di LncBook v2.0 è stato normalizzato in `lncbook_transcripts.parquet` da 3,68 MB e `lncbook_genes.parquet` da 0,79 MB; la conservazione da 46,78 MB `.csv.gz` a 23,92 MB Parquet con accesso colonnare istantaneo; `ensembl_paralogues.tsv` da 18,56 MB a 4,01 MB Parquet).
* **Decodifica del dump OHNOLOGS:** l'ispezione preliminare ha permesso di correggere due criticità strutturali dei file grezzi (assenza di intestazioni e compresenza di 36 specie con ID proteici `ENSP`), portando all'estrazione pulita di **4.602 geni WGD umani** su cromosomi canonici e **4.221 coppie WGD intra-umane** complete di coordinate, simboli genici, `strand`, famiglia (`OG...`) e classificazione del round di duplicazione (`r1`, `r2`, `both`, `syn`, `unk`, `htf`).
* **Costruzione e coerenza del controllo SSD:** interrogando gli archivi HTTPS ufficiali di Ensembl BioMart con filtro `biotype = protein_coding` su cromosomi canonici — e sottraendo tramite `anti-join` su `pair_key` tutte le coppie WGD — sono state isolate **127.578 coppie SSD pure**. Questo ordine di grandezza è pienamente allineato con i dataset di controllo utilizzati in letteratura dal gruppo di Torino (~122.800–122.900 coppie SSD in Mottes et al. 2021 e Agasso et al. 2025).
* **Nota critica emersa dal QC sulle SSD:** l'anteprima delle coppie SSD evidenzia la presenza di duplicazioni evolutivamente molto recenti (es. `NPIPA8–NPIPB2` con ultimo antenato comune `Homininae`), accanto a duplicazioni antiche (`Opisthokonta`, `Bilateria`). Poiché le 1R/2R-WGD risalgono a ~500 milioni di anni fa, questo dato conferma la necessità — già sottolineata da Mottes et al. (2021) — di stratificare nella fase successiva le coppie SSD per età evolutiva (escludendo le duplicazioni più giovani di *Sarcopterygii*).
* **Dataset opzionale non ancora incluso:** `lncrna_rbp_LncBook2.0.csv.gz` (interazioni lncRNA–proteine leganti l'RNA) risulta attualmente assente in `bronze/`; la pipeline lo gestisce in modo modulare e potrà integrarlo senza modificare i dataset già generati.

## 5. Conclusioni

Al termine della Fase 1 disponiamo di un Data Lake unificato, riproducibile e validato in formato Parquet su Google Drive, in cui l'universo dei **lncRNA umani (LncBook v2.0)**, i **4.602 geni / 4.221 coppie WGD (OHNOLOGS)** e le **127.578 coppie di controllo SSD pure (Ensembl)** condividono finalmente lo stesso sistema di coordinate genomiche GRCh38, gli stessi identificativi genici (`ENSG` / `HSALNG`) e la stessa informazione di orientamento (`strand`).

## 6. Prossimi Passi $\to$ Notebook 02 (`02_wgd_vs_ssd_evolution.ipynb`)

Con il layer `silver/` consolidato, il notebook successivo affronta la **Fase 2 — Identificazione Evolutiva degli Ohno-lncRNA**:
1. mappatura genomica dei lncRNA intragenici contenuti nei geni host `protein_coding` e confronto statistico **WGD vs SSD antiche** (estensione del modello intragenico di Agasso et al., 2025);
2. profilazione filostratigrafica indipendente tramite il nodo `17. Euteleostomi` di LncBook v2.0 (per quantificare anche i **lincRNA intergenici**);
3. verifica statistica del potenziale confondente della **lunghezza dei geni host** (Mantel-Haenszel + regressione logistica multivariata) e costruzione della curva di sensibilità dell'intersezione $A \cap B$, esportando le tre coorti ufficiali per le analisi di rete della Fase 3.